# Krok 11. Regresja liniowa i prognoza sprzedaży sklepu (jak w lekcji Kaggle „Time Series")
**Po co ten notebook.** Przenosimy na dane sklepu to, co pokazują lekcje Ryana Holbrooka: najpierw **trend** (zmienna czasu, wielomiany), potem **opóźnienia** (`Lag_1`), potem **sezonowość**. Każdy krok to regresja liniowa z coraz większym zestawem cech, a na końcu porównujemy ją z baseline'ami z kroku 10.

**Co modelujemy.** Jeden szereg: **tygodniowa sprzedaż całego sklepu** w sztukach (bez asortymentu 319). To inny poziom niż w kroku 10, gdzie prognozowaliśmy każdy towar osobno. Tu pytamy: czy prosta regresja na szeregu sklepu trafia w obrót tygodnia tak dobrze jak baseline_2?

**Podział czasu (jak w całym projekcie).**
- **Uczenie:** tygodnie 2023–2024 (przed `ORIGIN` = 2024-12-30).
- **Test:** 2025 i styczeń 2026 (57 tygodni). Model widzi je dopiero przy ocenie.
- Wybór parametrów (stopień wielomianu, liczba składowych sezonowych) robimy **bez rocznika testowego**: uczymy na 2023, sprawdzamy na 2024, dopiero potem dopasowujemy na obu latach.

**Kroki.**
| krok | model | cechy | co pokazuje |
|---|---|---|---|
| 1 | trend liniowy | zmienna czasu `t` | ogólny kierunek |
| 2 | wielomiany 2, 3 i 11 stopnia | `t`, `t²`, … | ryzyko przeuczenia |
| 3 | opóźnienie | `Lag_1` | zależność od poprzedniego tygodnia |
| 4 | trend + sezon + święta | `t` + sinusy i cosinusy roczne + zmienne wskaźnikowe | pełny model „rok z góry" |
| 5 | porównanie | wszystkie powyższe + baseline'y | czy to się opłaca |

**Ważne ograniczenie.** Dwa lata to dla regresji z sezonowością bardzo mało: każdy tydzień roku ma tylko dwie obserwacje, a święta ruchome (Wielkanoc) po jednej na rok. Dlatego model sezonowy opisujemy ostrożnie.

In [ ]:
# Ustawienia wspólne: moduł eda_wspolne.py leży obok tego notebooka
import os, sys
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import eda_wspolne as ew

ew.wejdz_do_korzenia()      # przechodzi do katalogu głównego projektu (tam, gdzie jest plik z danymi)
ew.styl()
print("Katalog projektu:", os.getcwd(), "| EDA używa danych do", ew.KONIEC_EDA.date(), "(rok testowy", ew.ROK_TEST, "pomijamy)")

## 1. Szereg tygodniowy sklepu i trend ze średniej ruchomej
#### Objaśnienie
**Po co.** Zaczynamy tak jak lekcja: budujemy szereg czasowy i oglądamy go. Jeden punkt to suma sprzedaży wszystkich towarów w tygodniu od poniedziałku. Pomarańczowa linia to trend ze **średniej ruchomej** z 52 tygodni (cały rok, więc sezonowość się uśrednia; okno wyśrodkowane, na brzegach wystarczy pół okna), dokładnie jak w pierwszym zadaniu ćwiczenia.
**Jak czytać wynik.** Trend pokazuje poziom bez sezonu. Pas po prawej stronie to rok testowy, którego model nie widzi. Duże piki to święta (Wielkanoc, Boże Narodzenie), a dolina w wakacje lub styczniu to sezon. Jeśli średnia ruchoma rośnie lub maleje, regresja z trendem ma co opisywać; jeśli jest płaska, trend nic nie da.
**Parametry:** `ORIGIN` (początek testu) i `OKNO_TRENDU_TYG`.

In [ ]:
ORIGIN = pd.Timestamp("2024-12-30")      # jak w kroku 10: testujemy od tego poniedziałku
OKNO_TRENDU_TYG = 52

KONIEC_DANYCH = pd.Timestamp(pd.read_parquet(ew.PLIK_REL, columns=["Data"])["Data"].max())
fa = ew.wczytaj_sprzedaz(do=KONIEC_DANYCH)
kal = ew.kalendarz(do=KONIEC_DANYCH)
dni = kal.index
pn0 = dni.min() + pd.Timedelta(days=(7 - dni.min().dayofweek) % 7)
pn_ost = dni.max() - pd.Timedelta(days=dni.max().dayofweek)
if dni.max().dayofweek < 5:
    pn_ost -= pd.Timedelta(days=7)
tygodnie = pd.date_range(pn0, pn_ost, freq="7D")
fa = fa[fa["Data"] <= pn_ost + pd.Timedelta(days=6)]
fa = fa.assign(Tyg=fa["Data"] - pd.to_timedelta(fa["Data"].dt.dayofweek, unit="D"))
y = fa.groupby("Tyg")[ew.MIARA].sum().reindex(tygodnie).fillna(0.0)
y.index = tygodnie
del fa

W = len(y)
uczenie = np.where(tygodnie < ORIGIN)[0]
test = np.where(tygodnie >= ORIGIN)[0]
print(f"Tygodni: {W} ({tygodnie[0].date()} … {tygodnie[-1].date()}) | uczenie: {len(uczenie)} | test: {len(test)}")
print(f"Średnia tygodniowa: 2023 = {y[tygodnie.year == 2023].mean():,.0f} | 2024 = {y[tygodnie.year == 2024].mean():,.0f} | test = {y.iloc[test].mean():,.0f} szt.".replace(",", " "))

trend_sr = y.rolling(window=OKNO_TRENDU_TYG, center=True, min_periods=OKNO_TRENDU_TYG // 2).mean()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y.index, y.values, color="0.65", lw=1.2, label="sprzedaż tygodniowa")
ax.plot(trend_sr.index, trend_sr.values, color=ew.POM, lw=3, label=f"trend: średnia ruchoma {OKNO_TRENDU_TYG} tyg.")
ax.axvspan(ORIGIN, y.index[-1] + pd.Timedelta(days=6), color="#e6e6e3", alpha=0.6, lw=0, label="rok testowy")
ax.set_title("Sprzedaż tygodniowa sklepu (bez AsId 319)"); ax.set_ylabel("sztuki / tydzień"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 2. Wspólne narzędzia: miary i cechy
#### Objaśnienie
**Po co.** Zanim zbudujemy modele, definiujemy jedną miarę oceny i jeden sposób składania cech, żeby każdy krok był liczony tak samo.
**Miary.**
- **WAPE** = suma |sprzedane − prognoza| / suma sprzedanych (jak w kroku 10, na sumie sklepu w tygodniu).
- **obciążenie** = suma prognoz / suma sprzedaży − 100%.
- Błąd liczymy też osobno w tygodniach wielkanocnych (k = −2…+1 względem Wielkanocy), świątecznych (ISO 51, 52, 1) i zwykłych.
**Cechy.**
- `t` = numer tygodnia dzielony przez 52 (czas w latach, dla stabilności liczbowej); wielomian to `t`, `t²`, …
- sezon roczny = pary `sin`, `cos` o okresie roku (tzw. szereg Fouriera): `K = 1` to jedna fala w roku, większe `K` pozwala na więcej garbów i dolin w roku;
- święta = zmienne wskaźnikowe (0/1): tydzień Wielkanocy, tydzień po niej, tygodnie Bożego Narodzenia (ISO 52) i Nowego Roku (ISO 1).
**Jak czytać wynik.** Powinno się wypisać, że w roku testowym są 4 tygodnie wielkanocne i 4 tygodnie okołoświąteczne (jak w kroku 10).

In [ ]:
from dateutil.easter import easter
from sklearn.linear_model import LinearRegression

nr_iso = np.asarray(tygodnie.isocalendar().week)
srodek = tygodnie + pd.Timedelta(days=3)                      # czwartek tygodnia: reprezentuje cały tydzień
faza = 2 * np.pi * np.asarray(srodek.dayofyear) / 365.25
t_lata = np.arange(W) / 52.0
pn_wlk = {r: pd.Timestamp(easter(r)) - pd.Timedelta(days=pd.Timestamp(easter(r)).dayofweek) for r in range(tygodnie.year.min(), tygodnie.year.max() + 1)}
k_wlk = np.array([(t - pn_wlk[r]).days // 7 for t, r in zip(tygodnie, np.asarray(tygodnie.year))])
t_wlk = (k_wlk >= -2) & (k_wlk <= 1)
t_bn = np.isin(nr_iso, [51, 52, 1])

def cechy(stopien=1, K=0, swieta=False):
    X = {}
    for p in range(1, stopien + 1):
        X[f"t^{p}"] = t_lata ** p
    for k in range(1, K + 1):
        X[f"sin{k}"] = np.sin(k * faza)
        X[f"cos{k}"] = np.cos(k * faza)
    if swieta:
        X["wielkanoc_k0"] = (k_wlk == 0).astype(float)
        X["wielkanoc_k1"] = (k_wlk == 1).astype(float)
        X["BN_tydz52"] = (nr_iso == 52).astype(float)
        X["NR_tydz1"] = (nr_iso == 1).astype(float)
    return pd.DataFrame(X, index=tygodnie)

def miary(rzecz, prog, maska=None):
    m = np.ones(len(rzecz), bool) if maska is None else maska
    r, p = np.asarray(rzecz)[m], np.asarray(prog)[m]
    return 100 * np.abs(r - p).sum() / r.sum(), 100 * (p.sum() - r.sum()) / r.sum()

def dopasuj_i_prognozuj(X, idx_ucz, idx_prog):
    mod = LinearRegression().fit(X.iloc[idx_ucz], y.values[idx_ucz])
    return mod, mod.predict(X.iloc[idx_prog])

def raport(nazwa, prog_test):
    w, b = miary(y.values[test], prog_test)
    wl, _ = miary(y.values[test], prog_test, t_wlk[test])
    wb, _ = miary(y.values[test], prog_test, t_bn[test])
    wp, _ = miary(y.values[test], prog_test, ~t_wlk[test] & ~t_bn[test])
    return {"model": nazwa, "WAPE_%": w, "obciazenie_%": b, "WAPE_wielkanoc_%": wl, "WAPE_BN_NR_%": wb, "WAPE_pozostale_%": wp}

print("Tygodnie testowe wielkanocne:", int(t_wlk[test].sum()), "| okołoświąteczne (ISO 51, 52, 1):", int(t_bn[test].sum()))
WYNIKI = []                  # tu zbieramy tabelę porównawczą
PROGNOZY = {}                # i prognozy tygodniowe każdego modelu

## 3. Krok 1: trend liniowy (zmienna czasu)
#### Objaśnienie
**Po co.** Najprostszy model z lekcji: `sprzedaż = waga × t + wyraz wolny`. Uczy się prostej przez sprzedaż tygodniową z 2023–2024 i przedłuża ją na rok testowy.
**Jak czytać wynik.**
- **Waga** to przyrost sprzedaży tygodniowej na rok (w sztukach). Dodatnia = sklep rośnie, ujemna = maleje.
- **Czerwona linia** to prognoza na 2025: prosta bez żadnych wahań. Błąd taki model ma dlatego duży: zakłada, że sezonu i świąt nie ma.
- To też najsłabsza poprzeczka: jeśli bardziej złożony model nie jest od niej lepszy, nie ma po co go używać.

In [ ]:
X1 = cechy(stopien=1)
mod1, prog1 = dopasuj_i_prognozuj(X1, uczenie, test)
dopasowanie1 = mod1.predict(X1.iloc[uczenie])
print(f"Model: sprzedaż = {mod1.coef_[0]:,.0f} × t(lata) + {mod1.intercept_:,.0f}  →  przyrost {mod1.coef_[0]:,.0f} szt. tygodniowo na rok".replace(",", " "))
WYNIKI.append(raport("1. trend liniowy", prog1)); PROGNOZY["trend liniowy"] = prog1

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y.index, y.values, color="0.65", lw=1.2, label="sprzedaż")
ax.plot(tygodnie[uczenie], dopasowanie1, color=ew.NIEB, lw=3, label="trend (dopasowanie)")
ax.plot(tygodnie[test], prog1, color="#c0392b", lw=3, label="trend (prognoza 2025)")
ax.set_title("Krok 1. Trend liniowy"); ax.set_ylabel("sztuki / tydzień"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()
print(pd.DataFrame(WYNIKI).round(1).to_string(index=False))

## 4. Krok 2: wielomiany i ryzyko przeuczenia
#### Objaśnienie
**Po co.** Lekcja pokazuje, że trend można zakrzywić, dodając `t²`, `t³`… To kusi, bo wielomian wysokiego stopnia dopasuje się do lat uczących bardzo dobrze. Sprawdzamy, co to daje **poza** tymi latami.
**Co robi kod.** Dla stopni 1, 2, 3 i 11 liczy błąd na latach uczących (dopasowanie) i na roku testowym (prognoza). Wielomian 11. stopnia to ten sam eksperyment, który w ćwiczeniu pokazuje, że prognoza „ucieka".
**Jak czytać wynik.**
- Błąd na uczeniu **zawsze maleje** ze stopniem (więcej parametrów = lepsze dopasowanie).
- Błąd na teście zwykle **rośnie**: wyrazy `t^11` poza zakresem uczącym przyjmują ogromne wartości, więc prognoza odlatuje (dużo za wysoko albo poniżej zera).
- Wniosek z lekcji: do prognozy używaj niskiego stopnia (1–2), a wahania w roku opisuj osobnymi cechami (sezon), nie wielomianem.

In [ ]:
wiersze = []
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y.index, y.values, color="0.75", lw=1.0, label="sprzedaż")
for stopien, kol_ in [(1, ew.NIEB), (2, ew.POM), (3, "#2a9d8f"), (11, "#c0392b")]:
    Xs = cechy(stopien=stopien)
    mod, prog = dopasuj_i_prognozuj(Xs, uczenie, test)
    dop = mod.predict(Xs.iloc[uczenie])
    wiersze.append({"stopien": stopien, "WAPE_uczenie_%": miary(y.values[uczenie], dop)[0], "WAPE_test_%": miary(y.values[test], prog)[0],
                    "min_prognozy": prog.min(), "max_prognozy": prog.max()})
    ax.plot(tygodnie[uczenie], dop, color=kol_, lw=1.5)
    ax.plot(tygodnie[test], prog, color=kol_, lw=2.5, label=f"stopień {stopien}")
    if stopien in (2, 3):
        WYNIKI.append(raport(f"2. wielomian {stopien}. stopnia", prog)); PROGNOZY[f"wielomian {stopien}"] = prog
ax.set_ylim(0, y.max() * 1.3)
ax.set_title("Krok 2. Wielomiany: dobre dopasowanie na uczeniu, kłopoty w prognozie"); ax.set_ylabel("sztuki / tydzień"); ax.legend(frameon=False, ncol=5)
plt.tight_layout(); plt.show()
print(pd.DataFrame(wiersze).round(1).to_string(index=False))
print("(min_prognozy i max_prognozy: najniższa i najwyższa prognozowana sprzedaż tygodniowa w roku testowym)")

## 5. Krok 3: opóźnienie (`Lag_1`)
#### Objaśnienie
**Po co.** Druga rodzina cech z lekcji. `Lag_1` to sprzedaż z poprzedniego tygodnia. Model `sprzedaż = waga × Lag_1 + wyraz wolny` pokazuje **zależność seryjną** (autokorelację): czy dobry tydzień zwykle ma po sobie dobry tydzień.
**Dwa sposoby użycia.**
- **Prognoza na tydzień do przodu (one-step):** w każdym tygodniu testowym znamy prawdziwą sprzedaż z poprzedniego tygodnia. To realna sytuacja, gdy prognozę odświeża się co tydzień.
- **Prognoza rekurencyjna na cały rok:** prawdziwych opóźnień nie znamy, więc za `Lag_1` podstawiamy własną poprzednią prognozę. Błąd się kumuluje i prognoza zwykle spłaszcza się do średniej.
**Jak czytać wynik.**
- Na wykresie opóźnień punkty układają się wzdłuż prostej, gdy sprzedaż zależy od poprzedniego tygodnia. W sklepie z sezonem i świętami zależność jest umiarkowana: święta „przeskakują" (po Wielkanocy sprzedaż spada, choć poprzedni tydzień był najwyższy).
- Prognoza na tydzień do przodu jest zwykle znacznie lepsza od prognozy na cały rok, bo ma świeżą informację. To nie jest porównywalne z innymi modelami z jednego punktu startu, więc w tabeli oznaczamy ją osobno.

In [ ]:
lag1 = y.shift(1)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(lag1.iloc[uczenie[1:]], y.iloc[uczenie[1:]], s=14, color="0.25")
wsp = np.polyfit(lag1.iloc[uczenie[1:]], y.iloc[uczenie[1:]], 1)
xs = np.linspace(lag1.iloc[uczenie[1:]].min(), lag1.iloc[uczenie[1:]].max(), 50)
ax.plot(xs, np.polyval(wsp, xs), color=ew.POM, lw=2.5)
ax.set_xlabel("Lag_1 (poprzedni tydzień)"); ax.set_ylabel("sprzedaż w tygodniu"); ax.set_title("Wykres opóźnień (lata uczące)")
plt.tight_layout(); plt.show()
print(f"Korelacja sprzedaży z poprzednim tygodniem (uczenie): {np.corrcoef(lag1.iloc[uczenie[1:]], y.iloc[uczenie[1:]])[0, 1]:.2f}")

XL = pd.DataFrame({"Lag_1": lag1.values}, index=tygodnie)
mod_l = LinearRegression().fit(XL.iloc[uczenie[1:]], y.values[uczenie[1:]])
prog_l = mod_l.predict(XL.iloc[test])                     # one-step: w każdym tygodniu znamy prawdziwy poprzedni
rek = []
ostatni = y.values[uczenie[-1]]
for _ in test:                                              # rekurencyjnie: za Lag_1 wstawiamy własną prognozę
    ostatni = mod_l.predict(pd.DataFrame({"Lag_1": [ostatni]}))[0]
    rek.append(ostatni)
rek = np.array(rek)
WYNIKI.append(raport("3a. Lag_1, tydzień do przodu", prog_l)); PROGNOZY["Lag_1 (tydzień do przodu)"] = prog_l
WYNIKI.append(raport("3b. Lag_1, rekurencyjnie na cały rok", rek)); PROGNOZY["Lag_1 (rekurencyjnie)"] = rek
print(pd.DataFrame(WYNIKI[-2:]).round(1).to_string(index=False))

## 6. Krok 4: trend + sezon roczny + święta
#### Objaśnienie
**Po co.** To odpowiednik tego, co w kroku 10 robi `baseline_2`, ale w jednej regresji na szeregu sklepu: trend opisuje poziom, **sinusy i cosinusy** roczne opisują sezon (np. lato słabsze, jesień mocniejsza), a **zmienne wskaźnikowe** (0/1) dodają stałą poprawkę w tygodniach świątecznych.
**Wybór parametrów bez roku testowego.** Liczba składowych sezonowych `K` i stopień trendu są wybierane na **walidacji**: uczymy na 2023 (52 tygodnie), sprawdzamy na 2024. Dopiero wybrane ustawienie dopasowujemy na całych 2023–2024 i prognozujemy 2025. Ze względu na tylko jeden rok uczący w walidacji trend liczymy tu liniowy.
**Jak czytać wynik.**
- Tabela walidacji pokazuje, ile składowych sezonu opłaca się użyć. Zbyt duże `K` zaczyna dopasowywać się do szumu jednego roku (WAPE walidacji rośnie).
- Współczynniki zmiennych wskaźnikowych to efekt święta w sztukach tygodniowo względem tygodnia zwykłego: dodatni dla tygodnia Wielkanocy, ujemny dla tygodnia po niej.
- Wykres: prognoza powinna powtarzać garby sezonu i pik Wielkanocy; tam, gdzie rozjeżdża się z czarną linią, jest niewyjaśniony popyt (np. inny układ dni handlowych).

In [ ]:
KANDYDACI_K = [0, 1, 2, 3, 4, 6, 8]
ucz23 = uczenie[tygodnie[uczenie].year == 2023]
wal24 = uczenie[tygodnie[uczenie].year == 2024]
wal = []
for K in KANDYDACI_K:
    Xk = cechy(stopien=1, K=K, swieta=True)
    _, p = dopasuj_i_prognozuj(Xk, ucz23, wal24)
    wal.append({"K": K, "liczba_cech": Xk.shape[1], "WAPE_walidacja_2024_%": miary(y.values[wal24], p)[0]})
wal = pd.DataFrame(wal)
K_WYBRANE = int(wal.loc[wal["WAPE_walidacja_2024_%"].idxmin(), "K"])
print(wal.round(1).to_string(index=False))
print("Wybrane K:", K_WYBRANE)

X4 = cechy(stopien=1, K=K_WYBRANE, swieta=True)
mod4, prog4 = dopasuj_i_prognozuj(X4, uczenie, test)
dop4 = mod4.predict(X4.iloc[uczenie])
wsp = pd.Series(mod4.coef_, index=X4.columns)
print("Efekty świąt (szt. tygodniowo względem zwykłego tygodnia):")
print(wsp[["wielkanoc_k0", "wielkanoc_k1", "BN_tydz52", "NR_tydz1"]].round(0).to_string())
WYNIKI.append(raport(f"4. trend + sezon (K={K_WYBRANE}) + święta", prog4)); PROGNOZY["trend + sezon + święta"] = prog4

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y.index, y.values, color="black", lw=1.6, label="sprzedaż")
ax.plot(tygodnie[uczenie], dop4, color=ew.NIEB, lw=2, label="dopasowanie")
ax.plot(tygodnie[test], prog4, color="#c0392b", lw=2.5, label="prognoza 2025")
for t in tygodnie[test][t_wlk[test]]:
    ax.axvspan(t, t + pd.Timedelta(days=7), color="#f3d9a4", alpha=0.5, lw=0)
ax.set_title("Krok 4. Trend + sezon roczny + święta (tło: tygodnie wielkanocne)"); ax.set_ylabel("sztuki / tydzień"); ax.legend(frameon=False, ncol=3)
plt.tight_layout(); plt.show()
print(pd.DataFrame(WYNIKI[-1:]).round(1).to_string(index=False))

## 7. Krok 4b: dołożenie opóźnienia do modelu sezonowego
#### Objaśnienie
**Po co.** Lekcje kończą się radą: najlepsze modele łączą cechy kroku czasowego i opóźnień. Dokładamy więc `Lag_1` do modelu z kroku 4.
**Jak to działa.**
- Wersja **na tydzień do przodu** używa prawdziwego poprzedniego tygodnia, więc jest realna tylko wtedy, gdy prognozę odświeżamy co tydzień.
- Wersja **rekurencyjna na rok** podstawia własne prognozy za opóźnienie.
**Jak czytać wynik.** Jeśli opóźnienie nie poprawia modelu sezonowego, to cały użyteczny sygnał jest już w trendzie, sezonie i świętach. Zwykle `Lag_1` pomaga tam, gdzie w tygodniu zdarzył się jednorazowy skok (np. dostawa, promocja), a ten model go kontynuuje w następnym tygodniu: na wykresie przesuwa prognozę w stronę ostatnio widzianej sprzedaży.

In [ ]:
X5 = X4.copy()
X5["Lag_1"] = y.shift(1).values
mod5 = LinearRegression().fit(X5.iloc[uczenie[1:]], y.values[uczenie[1:]])
prog5a = mod5.predict(X5.iloc[test])                          # tydzień do przodu (prawdziwy Lag_1)

rek5 = []
ostatni = y.values[uczenie[-1]]
for j in test:                                                # rekurencyjnie na cały rok
    x = X5.iloc[[j]].copy()
    x["Lag_1"] = ostatni
    ostatni = mod5.predict(x)[0]
    rek5.append(ostatni)
rek5 = np.array(rek5)
WYNIKI.append(raport("4b. sezon + święta + Lag_1, tydzień do przodu", prog5a)); PROGNOZY["sezon + święta + Lag_1 (tydzień do przodu)"] = prog5a
WYNIKI.append(raport("4c. sezon + święta + Lag_1, rekurencyjnie", rek5)); PROGNOZY["sezon + święta + Lag_1 (rekurencyjnie)"] = rek5
print("Waga Lag_1:", round(float(mod5.coef_[list(X5.columns).index('Lag_1')]), 2))
print(pd.DataFrame(WYNIKI[-2:]).round(1).to_string(index=False))

## 8. Porównanie ze wszystkimi modelami i z baseline'ami z kroku 10
#### Objaśnienie
**Po co.** Zestawiamy wszystko na jednej skali: obrót sklepu w tygodniu w roku testowym. Z kroku 10 bierzemy prognozy `baseline_0`, `baseline_1` i `baseline_2` (poziom każdego towaru razy mnożniki) i sumujemy je po towarach do poziomu sklepu.
**Uczciwość porównania.**
- Rekurencyjne i roczne prognozy (kroki 1, 2, 3b, 4, 4c) powstają z jednego punktu startu, tak samo jak baseline'y. Można je porównywać ze sobą.
- Prognozy „tydzień do przodu" (3a, 4b) znają prawdziwy poprzedni tydzień, więc są łatwiejsze; są w tabeli tylko dla porządku i nie wygrywają „konkursu rocznego".
- Baseline'y sumowane z towarów nie mają prognozy dla towarów, które pojawiły się dopiero w 2025 (kilka procent sprzedaży), więc są w tym porównaniu odrobinę poszkodowane. Regresja na szeregu sklepu widzi je w sumie.
**Jak czytać wynik.** Najniższy WAPE wśród prognoz rocznych wygrywa. Jeśli `baseline_2` jest wyraźnie lepszy od regresji, to rozbicie na towary i mnożniki z EDA daje więcej niż jedna regresja na sumie. Jeśli regresja jest bliska, to dla prognozy obrotu całego sklepu prosty model wystarcza, a rozbicie na towary jest potrzebne tylko do zamówień.

In [ ]:
plik_bl = "dane/interim/baseline_eda_tydzien.parquet"
if os.path.exists(plik_bl):
    bl = pd.read_parquet(plik_bl)
    bl_t = bl.groupby("Tydzien")[["Rzeczywiste", "baseline_0", "baseline_1", "baseline_2"]].sum().reindex(tygodnie[test]).fillna(0.0)
    for k in ["baseline_0", "baseline_1", "baseline_2"]:
        w, b = miary(y.values[test], bl_t[k].values)
        WYNIKI.append({**raport(k + " (suma towarów, krok 10)", bl_t[k].values)})
        PROGNOZY[k] = bl_t[k].values
    print("Kontrola: suma sprzedaży testowej z kroku 10 vs tu:", round(float(bl_t['Rzeczywiste'].sum())), "vs", round(float(y.values[test].sum())))
else:
    print("Brak", plik_bl, "- uruchom najpierw notebook 10, żeby dołożyć baseline'y do porównania.")

tab = pd.DataFrame(WYNIKI).set_index("model")
roczne = [m for m in tab.index if not m.startswith(("3a", "4b"))]
print(tab.round(1).to_string())
print()
print("Najlepszy model roczny (jeden punkt startu):", tab.loc[roczne, "WAPE_%"].idxmin(), f"({tab.loc[roczne, 'WAPE_%'].min():.1f} %)")

fig, ax = plt.subplots(figsize=(11, 4.4))
ax.plot(tygodnie[test], y.values[test], color="black", lw=2.2, label="rzeczywista")
kolory = {"trend liniowy": "0.7", "trend + sezon + święta": "#c0392b", "baseline_2": ew.NIEB, "baseline_0": ew.SZARY}
for k, kol_ in kolory.items():
    if k in PROGNOZY:
        ax.plot(tygodnie[test], PROGNOZY[k], color=kol_, lw=1.8, label=k)
for t in tygodnie[test][t_wlk[test]]:
    ax.axvspan(t, t + pd.Timedelta(days=7), color="#f3d9a4", alpha=0.5, lw=0)
ax.set_title("Rok testowy: sprzedaż sklepu i wybrane prognozy (tło: tygodnie wielkanocne)"); ax.set_ylabel("sztuki / tydzień"); ax.legend(frameon=False, ncol=3)
plt.tight_layout(); plt.show()

## 9. Zapis
#### Objaśnienie
**Po co.** Zapisujemy tabelę ocen (`dane/eda/wynik_11_regresja_metryki.csv`) i prognozy tygodniowe każdego modelu (`dane/interim/regresja_sklep_tydzien.csv`), żeby wrócić do nich w kolejnych krokach bez ponownego liczenia.
**Jak czytać wynik.** W pliku z prognozami jest po jednej kolumnie na model, więc łatwo narysować dowolne zestawienie albo policzyć błąd w wybranym tygodniu.

In [ ]:
ew.zapisz_wynik(tab.reset_index(), "wynik_11_regresja_metryki.csv", index=False)
os.makedirs("dane/interim", exist_ok=True)
out = pd.DataFrame({"Tydzien": tygodnie[test], "Rzeczywiste": y.values[test], **PROGNOZY})
out.to_csv("dane/interim/regresja_sklep_tydzien.csv", index=False)
print("Zapisano: dane/interim/regresja_sklep_tydzien.csv", out.shape)